# Pipeline node 01 — VLM zero-shot annotation

Production-parallel of notebook [`../02-vlm-zero-shot-annotation.ipynb`](../02-vlm-zero-shot-annotation.ipynb). No Gradio, no FiftyOne, no inline previews — reads raw images from S3, calls the VLM, writes a YOLO-format dataset back to S3.

**S3 contract**
- *Input:*  `s3://$BUCKET/$RAW_PREFIX/train/*.jpg`  (raw images; bootstrap once from `datasets/ppe/images/train/`)
- *Output:* `s3://$BUCKET/$DATASET_PREFIX/{images,labels}/train/*`  plus `dataset.yaml`

**Env vars expected** (set in the Elyra node config or the RHOAI data connection secret):
- `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_ENDPOINT`, `AWS_DEFAULT_REGION`, `AWS_S3_BUCKET`
- `VLM_ENDPOINT`, `VLLM_API_TOKEN` (optional)
- `RAW_PREFIX` (default `workshop-data`), `DATASET_PREFIX` (default `workshop-dataset`)

In [ ]:
%%capture
!pip install requests pillow tqdm boto3==1.35.55 botocore==1.35.55

In [ ]:
import base64
import json
import mimetypes
import os
import re
import shutil
from pathlib import Path

import boto3
import botocore
import requests
from PIL import Image
from tqdm.auto import tqdm

# 7-class PPE compliance taxonomy — must match 02-vlm-zero-shot-annotation.ipynb.
CLASSES = ["person", "helmet", "no-helmet", "vest", "no-vest", "gloves", "no-gloves"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

BUCKET         = os.environ["AWS_S3_BUCKET"]
RAW_PREFIX     = os.environ.get("RAW_PREFIX", "workshop-data").rstrip("/")
DATASET_PREFIX = os.environ.get("DATASET_PREFIX", "workshop-dataset").rstrip("/")
VLM_ENDPOINT   = os.environ["VLM_ENDPOINT"]

STAGE = Path("/tmp/ppe-annotate")
IN_DIR   = STAGE / "raw"
OUT_DIR  = STAGE / "yolo"
IMG_DIR  = OUT_DIR / "images" / "train"
LBL_DIR  = OUT_DIR / "labels" / "train"
for d in (IN_DIR, IMG_DIR, LBL_DIR):
    d.mkdir(parents=True, exist_ok=True)

session = boto3.session.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
s3 = session.resource(
    "s3",
    config=botocore.client.Config(signature_version="s3v4"),
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)
bucket = s3.Bucket(BUCKET)
print(f"S3 bucket: {BUCKET}; raw prefix: {RAW_PREFIX}/train; dataset prefix: {DATASET_PREFIX}")

## Download raw images from S3

In [ ]:
objects = list(bucket.objects.filter(Prefix=f"{RAW_PREFIX}/train/"))
print(f"Found {len(objects)} raw objects under s3://{BUCKET}/{RAW_PREFIX}/train/")

for obj in tqdm(objects, desc="download"):
    name = Path(obj.key).name
    if not name:
        continue
    bucket.download_file(obj.key, str(IN_DIR / name))

raw_files = sorted(p for p in IN_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(f"Downloaded {len(raw_files)} images -> {IN_DIR}")

## Call the VLM on each image and write YOLO-format labels

In [ ]:
# Structured PPE-compliance prompt — see 02-vlm-zero-shot-annotation.ipynb for rationale.
PROMPT = (
    "You are a workplace PPE compliance inspector. For every person visible in the image, "
    "produce detections by walking this checklist:\n"
    "\n"
    "STEP 1 \u2014 Find every person. Emit exactly one \"person\" detection per individual, "
    "boxed around their full body.\n"
    "\n"
    "STEP 2 \u2014 For each person, make three binary decisions and emit one detection per item:\n"
    "  HEAD:  Is the person wearing a hard hat / safety helmet?\n"
    "           YES  \u2192 emit \"helmet\"    boxed around the helmet\n"
    "           NO   \u2192 emit \"no-helmet\" boxed around the bare head\n"
    "  TORSO: Is the person wearing a high-visibility safety vest?\n"
    "           YES  \u2192 emit \"vest\"      boxed around the vest\n"
    "           NO   \u2192 emit \"no-vest\"   boxed around the bare torso\n"
    "  HANDS: Are the person's hands covered by work gloves?\n"
    "           YES  \u2192 emit \"gloves\"    boxed around the gloves\n"
    "           NO   \u2192 emit \"no-gloves\" boxed around the bare hands\n"
    "           (Skip this item if the hands are not clearly visible.)\n"
    "\n"
    "STEP 3 \u2014 Use ONLY these exact label strings. Never output synonyms like 'hardhat', "
    "'hard-hat', 'safety hat', 'reflective vest', 'no hat':\n"
    "  person, helmet, no-helmet, vest, no-vest, gloves, no-gloves\n"
    "\n"
    "STEP 4 \u2014 Return ONLY valid JSON, no prose, no markdown fences:\n"
    "{\n"
    '  "image_size": {"width": <int>, "height": <int>},\n'
    '  "detections": [\n'
    '    {"label": "<one of the 7 labels>",\n'
    '     "bbox_2d": [x1, y1, x2, y2], "confidence": <float 0-1>}\n'
    "  ]\n"
    "}\n"
    "Use absolute pixel coordinates: (x1,y1)=top-left, (x2,y2)=bottom-right.\n"
    "\n"
    "EXAMPLE \u2014 for one worker with helmet + gloves but no vest:\n"
    "{\n"
    '  "image_size": {"width": 1920, "height": 1080},\n'
    '  "detections": [\n'
    '    {"label": "person",   "bbox_2d": [420, 180, 720, 920], "confidence": 0.98},\n'
    '    {"label": "helmet",   "bbox_2d": [510, 200, 610, 270], "confidence": 0.95},\n'
    '    {"label": "no-vest",  "bbox_2d": [450, 310, 700, 600], "confidence": 0.88},\n'
    '    {"label": "gloves",   "bbox_2d": [470, 670, 530, 720], "confidence": 0.82}\n'
    "  ]\n"
    "}"
)


def auth_headers() -> dict:
    token = os.environ.get("VLLM_API_TOKEN") or os.environ.get("OPENAI_API_KEY")
    return {"Authorization": f"Bearer {token}"} if token else {}


def encode_image(path: Path) -> str:
    mime = mimetypes.guess_type(path.name)[0] or "image/jpeg"
    return f"data:{mime};base64,{base64.b64encode(path.read_bytes()).decode()}"


def get_model() -> str:
    r = requests.get(f"{VLM_ENDPOINT}/v1/models", timeout=30, headers=auth_headers())
    r.raise_for_status()
    return r.json()["data"][0]["id"]


def infer(path: Path, model: str) -> dict:
    payload = {
        "model": model,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT},
                {"type": "image_url", "image_url": {"url": encode_image(path)}},
            ],
        }],
        "max_tokens": 1024,
        "temperature": 0.2,
    }
    r = requests.post(
        f"{VLM_ENDPOINT}/v1/chat/completions",
        json=payload, timeout=180,
        headers={"Content-Type": "application/json", **auth_headers()},
    )
    r.raise_for_status()
    raw = r.json()["choices"][0]["message"]["content"]
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found: {raw[:200]}")
    return json.loads(match.group(0))


def write_yolo_label(parsed: dict, img_w: int, img_h: int, out_path: Path) -> int:
    n = 0
    with out_path.open("w") as f:
        for d in parsed.get("detections", []):
            lab = d.get("label")
            if lab not in CLASS_TO_IDX:
                continue
            x1, y1, x2, y2 = d["bbox_2d"]
            x1, x2 = sorted((max(0, min(img_w, x1)), max(0, min(img_w, x2))))
            y1, y2 = sorted((max(0, min(img_h, y1)), max(0, min(img_h, y2))))
            if x2 <= x1 or y2 <= y1:
                continue
            # YOLO format: class cx cy w h  (all normalized 0..1)
            cx = ((x1 + x2) / 2) / img_w
            cy = ((y1 + y2) / 2) / img_h
            w  = (x2 - x1) / img_w
            h  = (y2 - y1) / img_h
            f.write(f"{CLASS_TO_IDX[lab]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
            n += 1
    return n


model_id = get_model()
print("Serving model:", model_id)

total_boxes = 0
failures = []
for src in tqdm(raw_files, desc="annotate"):
    try:
        with Image.open(src) as im:
            img_w, img_h = im.size
        parsed = infer(src, model_id)
        # Mirror image to output dir (same filename) and write matching .txt.
        shutil.copy2(src, IMG_DIR / src.name)
        n = write_yolo_label(parsed, img_w, img_h, LBL_DIR / (src.stem + ".txt"))
        total_boxes += n
    except Exception as e:
        failures.append((src.name, str(e)))

print(f"Annotated {len(raw_files) - len(failures)}/{len(raw_files)} images; {total_boxes} total boxes.")
if failures[:3]:
    print("First failures:", failures[:3])

## Write dataset.yaml and upload the whole YOLO dataset to S3

The 40-image workshop dataset reuses `train` as the val split — see the root `CLAUDE.md` for the rationale.

In [ ]:
yaml_body = (
    "path: .\n"
    "train: images/train\n"
    "val: images/train\n"
    "names:\n" + "".join(f"  {i}: {c}\n" for i, c in enumerate(CLASSES))
)
(OUT_DIR / "dataset.yaml").write_text(yaml_body)


def upload_dir(local: Path, prefix: str) -> int:
    n = 0
    for p in local.rglob("*"):
        if p.is_file():
            key = f"{prefix}/{p.relative_to(local).as_posix()}"
            bucket.upload_file(str(p), key)
            n += 1
    return n


uploaded = upload_dir(OUT_DIR, DATASET_PREFIX)
print(f"Uploaded {uploaded} objects to s3://{BUCKET}/{DATASET_PREFIX}/")